# UrbanRay — AI reflection-correction research edition

This ZIP contains the AI-in-the-loop simulator, validation-selected residual model, audited test scores, offline dashboard, and original 10-scene exploratory results. **Do not treat any simulator label as an observed rooftop temperature or claim hackathon eligibility without checking the rules.**


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile, io
uploaded = files.upload()
choices = [name for name in uploaded if name.startswith('UrbanRay_') and name.endswith('.zip')]
if len(choices) != 1:
    raise ValueError('Upload exactly one complete UrbanRay project ZIP')
with zipfile.ZipFile(io.BytesIO(uploaded[choices[0]])) as z:
    z.extractall('/content')
project = Path('/content/UrbanRay_Latest_Complete')
print('Project found:', project.is_dir())
print('AI dashboard:', (project/'outputs/ai_upgrade/UrbanRay_AI_Explorer_OFFLINE.html').is_file())


In [ ]:
%cd /content/UrbanRay_Latest_Complete
!python -m pip install -q -r requirements.txt

In [ ]:
!PYTHONPATH=src python -m pytest -q tests

## AI reflection-correction model & offline dashboard
The learned target is *simulator-generated extra absorbed shortwave radiation*; no field temperature accuracy is established.


In [ ]:
from IPython.display import display
import pandas as pd, json
display(pd.read_csv('outputs/ai_upgrade/reflection_residual/candidate_heldout_radiation_scores.csv'))
print('Chosen mode:', json.load(open('outputs/ai_upgrade/reflection_residual/candidate_selection_audit.json'))['selected_by_validation'])


In [ ]:
from google.colab import files
files.download('outputs/ai_upgrade/UrbanRay_AI_Explorer_OFFLINE.html')  # Open it in your browser for an interactive offline demo


### Rebuild AI output (optional, re-trains on included simulator labels)
No external Earth Engine or weather download is required for these two operations.


In [ ]:
# !python train_reflection_residual.py --faces outputs/historical_ml/physics_training_faces.csv --out outputs/ai_upgrade/reflection_residual
# !python build_ai_dashboard.py


## View the already-completed ten-scene results

The 2026 numbers below compare a modeled emission-temperature proxy to satellite retrieved land-surface temperature. This is exploratory and does **not** establish individual-rooftop accuracy.

In [ ]:
import pandas as pd, json
from IPython.display import display
scenes = pd.read_csv('outputs/multidate_validation/scene_level_EXPLORATORY.csv')
metrics = json.load(open('outputs/multidate_validation/heldout/heldout_diagnostic.json'))
display(scenes)
print('Held-out raw metrics:', metrics['heldout_raw'])
print('Held-out bias-corrected metrics:', metrics['heldout_bias_corrected'])

In [ ]:
from IPython.display import Image, display
display(Image(filename='outputs/multidate_validation/audit/heldout_scene_comparison.png'))

## OPTIONAL: Plan or rerun selected dates

The 10-date outputs are already archived as summaries. Rerunning requires downloading historical Open-Meteo weather and significant compute. The original full weather/per-date simulation products were not included in the uploaded summaries.

In [ ]:
!python UrbanRay_Run_10Date_Validation.py --all --dry-run

To rerun one date, remove the leading `#` in the next cell's Python source and run it. To rerun all, use `--all`; this can be costly in Colab.

In [ ]:
# !python UrbanRay_Run_10Date_Validation.py --dates 2025-04-23

## OPTIONAL: Retrain the AI

This targets simulated multi-bounce shortwave absorption. It is **not** training a measured-temperature prediction model.

In [ ]:
# !python train_surrogate.py --faces outputs/historical_ml/physics_training_faces.csv --out outputs/ml_retrained